In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sdv.metadata import SingleTableMetadata
from sdv.single_table import CTGANSynthesizer
from sdmetrics.reports.single_table import QualityReport
from sdv.sampling import Condition

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

DATA_PATH = Path(r"D:\btp-ml-cyber-roi\data\processed\combined_enriched_core.csv")
if not DATA_PATH.exists():
    hits = list(Path.cwd().rglob("combined_enriched_core.csv"))
    if not hits:
        raise FileNotFoundError("combined_enriched_core.csv not found")
    DATA_PATH = hits[0]
print("Using:", DATA_PATH.resolve())


Using: D:\btp-ml-cyber-roi\data\processed\combined_enriched_core.csv


In [2]:
df = pd.read_csv(DATA_PATH)
print(f"Rows: {len(df)} | Columns: {df.shape[1]}")
print(json.dumps({"na_counts": df.isna().sum().to_dict()}, indent=2)[:1200])
df.head(3)


Rows: 3025 | Columns: 18
{
  "na_counts": {
    "Industry": 0,
    "Country": 0,
    "Year": 25,
    "Attack_Type": 0,
    "Canonical_Attack_Vector": 0,
    "Data_Type": 0,
    "Records_Compromised": 0,
    "Employee_Count": 0,
    "Security_Budget_Million_USD": 0,
    "Financial_Impact_Million_USD": 0,
    "Recovery_Time_Days": 0,
    "Incident_Severity": 25,
    "Baseline_Industry_Cost_Million_USD": 25,
    "Per_Record_Cost_USD": 25,
    "Estimated_Financial_Impact_Million_USD": 25,
    "Baseline_Source": 25,
    "PerRecord_Source": 3025,
    "Source_Tag": 0
  }
}


,Industry,Country,Year,Attack_Type,Canonical_Attack_Vector,Data_Type,Records_Compromised,Employee_Count,Security_Budget_Million_USD,Financial_Impact_Million_USD,Recovery_Time_Days,Incident_Severity,Baseline_Industry_Cost_Million_USD,Per_Record_Cost_USD,Estimated_Financial_Impact_Million_USD,Baseline_Source,PerRecord_Source,Source_Tag
0,Healthcare,United States,NaN,Ransomware,phishing,"PII,PHI,Financial",2400000,12500,15.0,125.0,45,NaN,NaN,NaN,NaN,NaN,NaN,RICH
1,Financial,United Kingdom,NaN,Data_Breach,unknown,"Financial,PII",890000,45000,45.0,89.0,21,NaN,NaN,NaN,NaN,NaN,NaN,RICH
2,Education,Canada,NaN,Ransomware,unknown,"PII,Academic_Records",567000,3400,2.1,34.5,18,NaN,NaN,NaN,NaN,NaN,NaN,RICH


In [3]:
CORE_COLS = [
    "Industry","Country","Year",
    "Attack_Type","Attack_Vector","Data_Type",
    "Records_Compromised","Employee_Count","Security_Budget_Million_USD",
    "Financial_Impact_Million_USD","Recovery_Time_Days","Incident_Severity",
    "Baseline_Industry_Cost_Million_USD","Per_Record_Cost_USD","Estimated_Financial_Impact_Million_USD",
]
data = df[[c for c in CORE_COLS if c in df.columns]].copy()

num_cols = [
    "Year","Records_Compromised","Employee_Count","Security_Budget_Million_USD",
    "Financial_Impact_Million_USD","Recovery_Time_Days","Incident_Severity",
    "Baseline_Industry_Cost_Million_USD","Per_Record_Cost_USD","Estimated_Financial_Impact_Million_USD"
]
for c in num_cols:
    if c in data.columns:
        data[c] = pd.to_numeric(data[c], errors="coerce")

cat_cols = ["Industry","Country","Attack_Type","Attack_Vector","Data_Type"]
for c in cat_cols:
    if c in data.columns:
        data[c] = data[c].astype("object")

critical = ["Records_Compromised","Employee_Count","Financial_Impact_Million_USD"]
before = len(data)
data = data.dropna(subset=[c for c in critical if c in data.columns])
print(f"Cleaned rows: {before} → {len(data)}")


Cleaned rows: 3025 → 3025


In [4]:
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data=data)

for c in cat_cols:
    if c in data.columns:
        metadata.update_column(column_name=c, sdtype="categorical")
for c in num_cols:
    if c in data.columns:
        metadata.update_column(column_name=c, sdtype="numerical")

metadata.validate()


In [5]:
SYN_ROWS = min(8 * len(data), 50_000)
PAC = 8
BATCH = 256

synth = CTGANSynthesizer(
    metadata,
    epochs=300,
    batch_size=BATCH,
    generator_dim=(256, 256),
    discriminator_dim=(256, 256),
    pac=PAC,
    verbose=True
)
synth.fit(data)

synthetic_v1 = synth.sample(SYN_ROWS)
print("v1 synthetic rows:", len(synthetic_v1))


d:\btp-ml-cyber-roi\.venv310\lib\site-packages\sdv\single_table\base.py:86: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
d:\btp-ml-cyber-roi\.venv310\lib\site-packages\ctgan\synthesizers\_utils.py:16: FutureWarning: `cuda` parameter is deprecated and will be removed in a future release. Please use `enable_gpu` instead.
  warnings.warn(
Gen. (-0.47) | Discrim. (-0.43): 100%|██████████| 300/300 [04:08<00:00,  1.21it/s]


v1 synthetic rows: 24200


In [6]:
def quick_compare(real, synth, col):
    a = pd.to_numeric(real[col], errors="coerce").dropna()
    b = pd.to_numeric(synth[col], errors="coerce").dropna()
    if a.empty or b.empty:
        print(f"{col}: skip (empty)")
        return
    print(f"{col}: real mean={a.mean():.3g}, std={a.std():.3g} | synth mean={b.mean():.3g}, std={b.std():.3g}")

for col in ["Records_Compromised","Employee_Count","Security_Budget_Million_USD",
            "Financial_Impact_Million_USD","Per_Record_Cost_USD","Baseline_Industry_Cost_Million_USD"]:
    if col in data.columns and col in synthetic_v1.columns:
        quick_compare(data, synthetic_v1, col)


Records_Compromised: real mean=5.14e+05, std=3.81e+05 | synth mean=4.44e+05, std=3.47e+05
Employee_Count: real mean=1.43e+04, std=1.36e+04 | synth mean=1.41e+04, std=1.39e+04
Security_Budget_Million_USD: real mean=13.9, std=9.5 | synth mean=13.7, std=10.7
Financial_Impact_Million_USD: real mean=51.5, std=33.9 | synth mean=65.4, std=35.7
Per_Record_Cost_USD: real mean=100, std=0 | synth mean=100, std=0
Baseline_Industry_Cost_Million_USD: real mean=4.65, std=1.58 | synth mean=4.69, std=1.34


In [7]:
def winsorize(s, p=0.005):
    lo, hi = s.quantile([p, 1-p])
    return s.clip(lo, hi)

for c in ["Records_Compromised","Financial_Impact_Million_USD","Security_Budget_Million_USD","Employee_Count"]:
    if c in data.columns:
        data[c] = winsorize(pd.to_numeric(data[c], errors="coerce"))

PAC = 8
BATCH = 256

synth = CTGANSynthesizer(
    metadata,
    epochs=600,
    batch_size=BATCH,
    generator_dim=(512, 512),
    discriminator_dim=(512, 512),
    pac=PAC,
    verbose=True
)
synth.fit(data)


d:\btp-ml-cyber-roi\.venv310\lib\site-packages\ctgan\synthesizers\_utils.py:16: FutureWarning: `cuda` parameter is deprecated and will be removed in a future release. Please use `enable_gpu` instead.
  warnings.warn(
Gen. (-0.14) | Discrim. (-0.34): 100%|██████████| 600/600 [14:52<00:00,  1.49s/it]


In [8]:
vc = (data.groupby(["Industry","Attack_Type"]).size()
        .sort_values(ascending=False))
top = vc.head(15)
rare = vc[vc < vc.median()]

rows_total = min(8 * len(data), 50_000)
plan = []
for (ind, atk), cnt in top.items():
    n = max(int(rows_total * (cnt / len(data)) * 1.10), 50)
    plan.append({"Industry": ind, "Attack_Type": atk, "num_rows": n})
for (ind, atk), cnt in rare.items():
    plan.append({"Industry": ind, "Attack_Type": atk, "num_rows": 30})

conds_df = pd.DataFrame(plan)

conditions = [
    Condition(column_values={"Industry": str(r["Industry"]), "Attack_Type": str(r["Attack_Type"])},
              num_rows=int(r["num_rows"]))
    for _, r in conds_df.iterrows()
]

synthetic_v2 = synth.sample_from_conditions(conditions=conditions, max_tries_per_batch=200)
target_total = rows_total
if len(synthetic_v2) < target_total:
    extra = synth.sample(target_total - len(synthetic_v2))
    synthetic_v2 = pd.concat([synthetic_v2, extra], ignore_index=True)

print("v2 synthetic rows:", len(synthetic_v2))


Sampling conditions: 100%|██████████| 11708/11708 [01:55<00:00, 101.44it/s]


v2 synthetic rows: 24200


In [9]:
def corr_diff(real, synth, cols):
    corr_real = real[cols].corr(method="spearman")
    corr_synth = synth[cols].corr(method="spearman")
    delta = (corr_real - corr_synth).abs()
    print("Mean abs correlation difference:", round(delta.values.mean(), 3))
    return delta

corr_cols = ["Employee_Count","Security_Budget_Million_USD","Records_Compromised","Financial_Impact_Million_USD"]
_ = corr_diff(data, synthetic_v2, corr_cols)

report = QualityReport()
report.generate(real_data=data, synthetic_data=synthetic_v2, metadata=metadata.to_dict())
print(f"\nOverall SDV Quality Score: {report.get_score():.3f}")
props = report.get_properties()
display(props)


Mean abs correlation difference: 0.054
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 14/14 [00:00<00:00, 199.30it/s]|
Column Shapes Score: 82.51%

(2/2) Evaluating Column Pair Trends: |██████████| 91/91 [00:00<00:00, 144.78it/s]|
Column Pair Trends Score: 80.29%

Overall Score (Average): 81.4%


Overall SDV Quality Score: 0.814


,Property,Score
0,Column Shapes,0.825112
1,Column Pair Trends,0.802894


In [10]:
out_dir = Path(r"D:\btp-ml-cyber-roi\data\model_ready")
out_dir.mkdir(parents=True, exist_ok=True)

clean_path = out_dir / "combined_clean.csv"
v1_path    = out_dir / "synthetic_samples_ctgan_v1.csv"
v2_path    = out_dir / "synthetic_samples_ctgan_v2.csv"
model_path = out_dir / "ctgan_synthesizer.pkl"
report_path= out_dir / "sdv_quality_report_v2.pkl"

data.to_csv(clean_path, index=False)
synthetic_v1.to_csv(v1_path, index=False)
synthetic_v2.to_csv(v2_path, index=False)

# Persist model and report
synth.save(model_path)
report.save(report_path)

print("Saved:")
print(" -", clean_path)
print(" -", v1_path)
print(" -", v2_path)
print(" -", model_path)
print(" -", report_path)


Saved:
 - D:\btp-ml-cyber-roi\data\model_ready\combined_clean.csv
 - D:\btp-ml-cyber-roi\data\model_ready\synthetic_samples_ctgan_v1.csv
 - D:\btp-ml-cyber-roi\data\model_ready\synthetic_samples_ctgan_v2.csv
 - D:\btp-ml-cyber-roi\data\model_ready\ctgan_synthesizer.pkl
 - D:\btp-ml-cyber-roi\data\model_ready\sdv_quality_report_v2.pkl
